# Sentence Model - Original data

This notebook will train the detection model at the sentence embedding data for the original non-paraphrased texts. The inputs should be three sentence embedding files.

# Load data

In [1]:
import numpy as np
import tensorflow as tf
import pickle
import pandas as pd
import keras

# Class definition

In [2]:
# from keras.backend import dropout
def positional_encoding(length, depth):
  depth = depth/2

  positions = np.arange(length)[:, np.newaxis]     # (seq, 1)
  depths = np.arange(depth)[np.newaxis, :]/depth   # (1, depth)

  angle_rates = 1 / (10000**depths)         # (1, depth)
  angle_rads = positions * angle_rates      # (pos, depth)

  pos_encoding = np.concatenate(
      [np.sin(angle_rads), np.cos(angle_rads)],
      axis=-1)

  return tf.cast(pos_encoding, dtype=tf.float32)


class PositionalEmbedding(tf.keras.layers.Layer):
  def __init__(self, d_model):
    super().__init__()
    self.d_model = d_model
    self.pos_encoding = positional_encoding(length=2048, depth=d_model)

  def call(self, x):
    length = tf.shape(x)[1]
    x *= tf.math.sqrt(tf.cast(self.d_model, tf.float32))
    x = x + self.pos_encoding[tf.newaxis, :length, :]
    return x


class MHAttention(tf.keras.layers.Layer):
  def __init__(self, **kwargs):
    super().__init__()
    self.mha = tf.keras.layers.MultiHeadAttention(**kwargs)
    self.layernorm = tf.keras.layers.LayerNormalization()
    self.add = tf.keras.layers.Add()

  def call(self, x):
      attn_output = self.mha(
          query=x,
          value=x,
          key=x)
      x = self.add([x, attn_output])
      x = self.layernorm(x)
      return x


class FeedForward(tf.keras.layers.Layer):
  def __init__(self, d_model, dff, dropout_rate=0.1):
    super().__init__()
    self.seq = tf.keras.Sequential([
      tf.keras.layers.Dense(dff, activation='relu'),
      tf.keras.layers.Dense(d_model),
      tf.keras.layers.Dropout(dropout_rate)
    ])
    self.add = tf.keras.layers.Add()
    self.layer_norm = tf.keras.layers.LayerNormalization()

  def call(self, x):
    x = self.add([x, self.seq(x)])
    x = self.layer_norm(x)
    return x


class EncoderLayer(tf.keras.layers.Layer):
  def __init__(self,*, d_model, num_heads, dff, dropout_rate=0.1):
    super().__init__()

    self.self_attention = MHAttention(
        num_heads=num_heads,
        key_dim=d_model,
        dropout=dropout_rate)

    self.ffn = FeedForward(d_model, dff)

  def call(self, x):
    x = self.self_attention(x)
    x = self.ffn(x)
    return x


class Encoder(tf.keras.layers.Layer):
  def __init__(self, *, num_layers, d_model, num_heads, dff, dropout_rate=0.1):
    super().__init__()

    self.d_model = d_model
    self.num_layers = num_layers

    # self.pos_embedding = PositionalEmbedding(d_model=d_model)

    self.enc_layers = [
        EncoderLayer(d_model=d_model,
                     num_heads=num_heads,
                     dff=dff,
                     dropout_rate=dropout_rate)
        for _ in range(num_layers)]

    self.dropout = tf.keras.layers.Dropout(dropout_rate)

  def call(self, x):
    # `x` is token-IDs shape: (batch, seq_len)
    # x = self.pos_embedding(x)  # Shape `(batch_size, seq_len, d_model)`.

    # Add dropout.
    x = self.dropout(x)

    for i in range(self.num_layers):
      x = self.enc_layers[i](x)

    return x  # Shape `(batch_size, seq_len, d_model)`.


class CrossAttention(tf.keras.layers.Layer):
  def __init__(self, **kwargs):
    super().__init__()
    self.mha = tf.keras.layers.MultiHeadAttention(**kwargs)
    self.layernorm = tf.keras.layers.LayerNormalization()
    self.add = tf.keras.layers.Add()

  def call(self, x, y):
      attn_output = self.mha(
          query=x,
          value=y,
          key=y)
      x = self.add([x, attn_output])
      x = self.layernorm(x)
      return x


class DecoderLayer(tf.keras.layers.Layer):
  def __init__(self,
               *,
               d_model,
               num_heads,
               dff,
               dropout_rate=0.1):
    super(DecoderLayer, self).__init__()

    self.self_attention = MHAttention(
        num_heads=num_heads,
        key_dim=d_model,
        dropout=dropout_rate)

    self.cross_attention = CrossAttention(
        num_heads=num_heads,
        key_dim=d_model,
        dropout=dropout_rate)

    self.ffn = FeedForward(d_model, dff)

  def call(self, x, y):
    x = self.self_attention(x=x)
    x = self.cross_attention(x=x, y=y)

    x = self.ffn(x)  # Shape `(batch_size, seq_len, d_model)`.
    return x


class Decoder(tf.keras.layers.Layer):
  def __init__(self, *, num_layers, d_model, num_heads, dff, dropout_rate=0.1):
    super(Decoder, self).__init__()

    self.d_model = d_model
    self.num_layers = num_layers

    # self.pos_embedding = PositionalEmbedding(d_model=d_model)
    self.dropout = tf.keras.layers.Dropout(dropout_rate)
    self.dec_layers = [
        DecoderLayer(d_model=d_model, num_heads=num_heads,
                     dff=dff, dropout_rate=dropout_rate)
        for _ in range(num_layers)]

  def call(self, x, y):
    # `x` is token-IDs shape (batch, target_seq_len)
    # x = self.pos_embedding(x)  # (batch_size, target_seq_len, d_model)

    x = self.dropout(x)

    for i in range(self.num_layers):
      x  = self.dec_layers[i](x, y)

    # The shape of x is (batch_size, target_seq_len, d_model).
    return x


class MetricLayer(tf.keras.layers.Layer):
  def __init__(self, *, num_layers, d_model, num_heads, dff, num_denses, dropout_rate=0.1):
      super().__init__()
      self.encoder = Encoder(
          num_layers=num_layers,
          d_model=d_model,
          num_heads=num_heads,
          dff=dff,
          dropout_rate=dropout_rate
      )

      self.decoder = Decoder(
          num_layers=num_layers,
          d_model=d_model,
          num_heads=num_heads,
          dff=dff,
          dropout_rate=dropout_rate
      )

      self.denses = [FeedForward(d_model, dff, dropout_rate) for _ in range(num_denses)]
      self.dropouts = [tf.keras.layers.Dropout(dropout_rate) for _ in range(num_denses)]
      self.final_layer = tf.keras.layers.Dense(1, activation='sigmoid')

  def call(self, x, y, training=None):
      y = self.encoder(y)
      x = self.decoder(x, y)

      for dense, dropout in zip(self.denses, self.dropouts):
          x = dense(x)
          x = dropout(x, training=training)

      x = tf.math.reduce_sum(x, axis=1)
      x = self.final_layer(x)
      return x
    
class DistanceLayer(tf.keras.layers.Layer):
  def __init__(self, metric_layer, **kwargs):
      super().__init__(**kwargs)
      self.metric_layer = metric_layer

  def call(self, anchor, positive, negative):
      ap_distance = self.metric_layer(anchor, positive) #tf.reduce_sum(tf.square(anchor - positive), -1)
      an_distance = self.metric_layer(anchor, negative) #tf.reduce_sum(tf.square(anchor - negative), -1)
      return (ap_distance, an_distance)


class TripletModel(tf.keras.Model):
  def __init__(self, siamese_network, margin=0.5):
      super().__init__()
      self.siamese_network = siamese_network
      self.margin = margin
      self.loss_tracker = tf.keras.metrics.Mean(name="loss")

  def call(self, inputs):
      if isinstance(inputs, tuple):
          inputs = inputs[0]
      return self.siamese_network(inputs)
  def train_step(self, data):
      with tf.GradientTape() as tape:
          loss = self._compute_loss(data)
      gradients = tape.gradient(loss, self.siamese_network.trainable_weights)
      self.optimizer.apply_gradients(
          zip(gradients, self.siamese_network.trainable_weights)
      )
      self.loss_tracker.update_state(loss)
      return {"loss": self.loss_tracker.result()}

  def test_step(self, data):
      loss = self._compute_loss(data)
      self.loss_tracker.update_state(loss)
      return {"loss": self.loss_tracker.result()}

  def _compute_loss(self, data):
      if isinstance(data, tuple):
          data = data[0]   # unpack x from (x,) or (x, y)
      ap_distance, an_distance = self.siamese_network(data)
      loss = ap_distance - an_distance
      loss = tf.maximum(loss + self.margin, 0.0)
      return loss
  @property
  def metrics(self):
      # We need to list our metrics here so the `reset_states()` can be
      # called automatically.
      return [self.loss_tracker]


from sklearn.metrics import f1_score

def f1(x,y,thres):
    yx = np.zeros(x.shape[0])
    yy = np.ones(y.shape[0])
    yhx = np.zeros(x.shape[0])
    yhx[x > thres] = 1
    yhy = np.zeros(y.shape[0])
    yhy[y > thres] = 1
    return f1_score(np.concatenate([yx,yy]), np.concatenate([yhx,yhy]))

## Function to build and evaluate model

In [3]:
def run_model(input_shape, n_lower_layers, n_upper_layers, answers, gpt1, gpt2, epochs=50, seeds=(1111,2222,3333,4444,5555,6666,7777,8888,9999,1010)):
  valid_accuracies = []
  valid_f1s = []
  test_accuracies = []
  test_f1s = []

  for seed in seeds:
    np.random.seed(seed)

    #train-test-validation split
    sampling_indices = np.random.uniform(0,1,answers.shape[0])
    train_answers = answers[sampling_indices < 0.8]
    train_gpt1 = gpt1[sampling_indices < 0.8]
    train_gpt2 = gpt2[sampling_indices < 0.8]
    valid_answers = answers[(0.8 <= sampling_indices) & (sampling_indices < 0.9)]
    valid_gpt1 = gpt1[(0.8 <= sampling_indices) & (sampling_indices < 0.9)]
    valid_gpt2 = gpt2[(0.8 <= sampling_indices) & (sampling_indices < 0.9)]
    test_answers = answers[sampling_indices >= 0.9]
    test_gpt1 = gpt1[sampling_indices >= 0.9]
    test_gpt2 = gpt2[sampling_indices >= 0.9]

    #build model
    emb_shape = input_shape
    metric_layer = MetricLayer(num_layers=n_lower_layers,
                              d_model=input_shape,
                              num_heads=4,
                              dff=input_shape,
                              num_denses=n_upper_layers)

    anchor_input = tf.keras.layers.Input(name="anchor", shape=[pad_length, emb_shape])    
    positive_input = tf.keras.layers.Input(name="positive", shape=[pad_length,emb_shape])
    negative_input = tf.keras.layers.Input(name="negative", shape=[pad_length,emb_shape])
    distances = DistanceLayer(metric_layer)(
        anchor_input,
        positive_input,
        negative_input
    )
    distance_network = tf.keras.Model(
        inputs=[anchor_input, positive_input, negative_input], outputs=distances
    )
    triplet_model = TripletModel(distance_network, margin=0.25)

    #training
    train_p1 = np.vstack([train_gpt1, train_gpt2])
    train_p2 = np.vstack([train_gpt2, train_gpt1])
    train_ng = np.vstack([train_answers, train_answers])
    triplet_model.compile(optimizer=tf.keras.optimizers.Adam(2e-5), weighted_metrics=[])
    triplet_model.fit([train_p1, train_p2, train_ng],
                      epochs=3, batch_size=16,
                      validation_data=([valid_gpt1, valid_gpt2, valid_answers],))
    triplet_model.compile(optimizer=tf.keras.optimizers.Adam(1e-6), weighted_metrics=[])
    triplet_model.fit([train_p1, train_p2, train_ng],
                      epochs=2, batch_size=16,
                      validation_data=([valid_gpt1, valid_gpt2, valid_answers],))
    #valid accuracy
    valid_gpt_distances, valid_answer_distances = distance_network.predict([valid_gpt1, valid_gpt2, valid_answers], batch_size=1024)
    valid_accuracies.append((valid_gpt_distances < valid_answer_distances).mean())

    #tune threshold
    thresholds = []
    perf = []
    for thres in np.arange(0,2,0.01):
      thresholds.append(thres)
      perf.append(f1(valid_gpt_distances.flatten(), valid_answer_distances.flatten(), thres))
    triplet_thres = thresholds[perf.index(max(perf))]

    #valid f1
    valid_f1s.append(max(perf))

    #test accuracy
    test_gpt_distances, test_answer_distances = distance_network.predict([test_gpt1, test_gpt2, test_answers], batch_size=1024)
    test_accuracies.append((test_gpt_distances < test_answer_distances).mean())

    #test f1
    test_f1s.append(f1(test_gpt_distances.flatten(), test_answer_distances.flatten(), triplet_thres))
    print(valid_accuracies, test_accuracies, valid_f1s, test_f1s)
    #clear memory
    keras.backend.clear_session()
  return valid_accuracies, test_accuracies, valid_f1s, test_f1s

# Load Data

Like the other notebooks, make sure to load sentence embedding data from the same embedding model here.

In [4]:
dataset = 'SQUAD_GPT4'
pad_length = 5

with open(f'/home/aanis/Metric-Models-for-Detection-of-LLM-Texts/Data/{dataset}_answer_sentence_embs.obj', 'rb') as f:
  answers = pickle.load(f)
  answers = tf.keras.preprocessing.sequence.pad_sequences(answers, maxlen=pad_length, padding='post', dtype='float32')
with open(f'/home/aanis/Metric-Models-for-Detection-of-LLM-Texts/Data/{dataset}_gpt1_sentence_embs.obj', 'rb') as f:
  gpt1 = pickle.load(f)
  gpt1 = tf.keras.preprocessing.sequence.pad_sequences(gpt1, maxlen=pad_length, padding='post', dtype='float32')
with open(f'/home/aanis/Metric-Models-for-Detection-of-LLM-Texts/Data/{dataset}_gpt2_sentence_embs.obj', 'rb') as f:
  gpt2 = pickle.load(f)
  gpt2 = tf.keras.preprocessing.sequence.pad_sequences(gpt2, maxlen=pad_length, padding='post', dtype='float32')

In [5]:
print("answers:", np.array(answers).shape, answers.dtype)
print("gpt1:", np.array(gpt1).shape, gpt1.dtype)
print("gpt2:", np.array(gpt2).shape, gpt2.dtype)

print("one answer sample shape:", np.array(answers[0]).shape)
print("one gpt1 sample shape:", np.array(gpt1[0]).shape)

answers_small = answers[:500]
gpt1_small = gpt1[:500]
gpt2_small = gpt2[:500]

print("new answer:", np.array(answers_small).shape)
print("new gpt1:", np.array(gpt1_small).shape)



answers: (4306, 5, 768) float32
gpt1: (4306, 5, 768) float32
gpt2: (4306, 5, 768) float32
one answer sample shape: (5, 768)
one gpt1 sample shape: (5, 768)
new answer: (500, 5, 768)
new gpt1: (500, 5, 768)


# Run Model

In [ ]:
input_shape = 768
n_lower_layers = 3
n_upper_layers = 3

valid_accuracies, test_accuracies, valid_f1s, test_f1s = run_model(input_shape, n_lower_layers, n_upper_layers,
                                                                   answers_small, gpt1_small, gpt2_small, 50)

Epoch 1/3


In [6]:
valid_accuracies, test_accuracies, valid_f1s, test_f1s = run_model(
    input_shape=768,
    n_lower_layers=1,
    n_upper_layers=1,
    answers=answers[:200],
    gpt1=gpt1[:200],
    gpt2=gpt2[:200],
    epochs=1,
    seeds=(1111,)
)

Epoch 1/3
 1/21 ━━━━━━━━━━━━━━━━━━━━ 2:51:01 513s/step - loss: 0.2297

KeyboardInterrupt: 

In [ ]:
np.mean(valid_accuracies), np.mean(test_accuracies), np.mean(valid_f1s), np.mean(test_f1s)

In [24]:
print("answers:", np.array(answers).shape, answers.dtype)
print("gpt1:", np.array(gpt1).shape, gpt1.dtype)
print("gpt2:", np.array(gpt2).shape, gpt2.dtype)

print("one answer sample shape:", np.array(answers[0]).shape)
print("one gpt1 sample shape:", np.array(gpt1[0]).shape)

answers: (4306, 5, 768) float32
gpt1: (4306, 5, 768) float32
gpt2: (4306, 5, 768) float32
one answer sample shape: (5, 768)
one gpt1 sample shape: (5, 768)
